# Градиентный бустинг над деревьями

Учебный ноутбук по материалам семинара № 9 «Градиентный бустинг» (ФКН ВШЭ) и обсуждения функционального градиентного спуска.

**Цель:** понять, почему модель складывает деревья, что означает псевдоостаток, как выбор функции потерь меняет обучение и чем бустинг отличается от случайного леса.

## Содержание
1. [Идея и сумма моделей](#1-идея-и-сумма-моделей)
2. [Антиградиент и скалярное произведение](#2-антиградиент-и-скалярное-произведение)
3. [Потери и псевдоостатки](#3-потери-и-псевдоостатки)
4. [Алгоритм и пример на Python](#4-алгоритм-и-пример-на-python)
5. [Глубина деревьев и контроль переобучения](#5-глубина-деревьев-и-контроль-переобучения)
6. [Бустинг и случайный лес: сложность](#6-бустинг-и-случайный-лес-сложность)
7. [XGBoost, LightGBM, CatBoost](#7-xgboost-lightgbm-catboost)

## 1. Идея и сумма моделей

Пусть есть пары $(x_i,y_i)$ и функция потерь $L(y,z)$, где $z$ — **числовой выход модели**. Минимизируем эмпирический риск:

$$Q(F)=\frac1n\sum_{i=1}^n L(y_i,F(x_i)).$$

Начинаем с простой константы $F_0(x)=\arg\min_c\sum_i L(y_i,c)$. Для квадратичной потери $L(y,z)=\tfrac12(y-z)^2$ это среднее $\bar y$.

На каждом шаге добавляем дерево как поправку:

$$F_m(x)=F_{m-1}(x)+\eta\gamma_m h_m(x),\qquad
F_M(x)=F_0(x)+\sum_{m=1}^M\eta\gamma_m h_m(x).$$

Здесь $\eta$ — темп обучения, а $\gamma_m$ — возможная оптимальная длина шага. Если она не подбирается отдельно, её эффект можно включить в значения листьев. Например, базовый прогноз 100 и поправки $+20,-5,+2$ дают 117. Дерево учится исправлять текущую композицию, поэтому возникает **сумма**.

## 2. Антиградиент и скалярное произведение

На обучающей выборке значения функции образуют вектор $\mathbf f=(F(x_1),\dots,F(x_n))$. Для $Q=\frac1n\sum_iL(y_i,f_i)$ производная по координате равна $\partial Q/\partial f_i=\frac1n\partial L(y_i,f_i)/\partial f_i$.

Если положить $g_i=\partial L(y_i,f_i)/\partial f_i$ и определить скалярное произведение $\langle u,v\rangle_n=\frac1n\sum_i u_iv_i$, то малое добавление $\varepsilon h$ даёт

$$Q(F+\varepsilon h)\approx Q(F)+\varepsilon\langle g,h\rangle_n.$$

Направление $h=-g$ уменьшает риск в первом приближении: $\langle g,-g\rangle_n=-\|g\|_n^2$. Поэтому цели для нового дерева — **псевдоостатки**

$$s_i=-\left.\frac{\partial L(y_i,z)}{\partial z}\right|_{z=F_{m-1}(x_i)}.$$

Множитель $1/n$ уже учтён в скалярном произведении: так не возникает противоречия между градиентом среднего риска и формулой для $s_i$. На конечной выборке хватает производных по $n$ числам. Производная Фреше нужна для строгой записи, когда аргумент риска — целая функция: $Q(F+h)=Q(F)+DQ[F](h)+o(\|h\|)$, причём в подходящем гильбертовом пространстве $DQ[F](h)=\langle\nabla Q(F),h\rangle$. Для вычисления алгоритма знать теорему Фреше не требуется.

**Оговорка:** направление самого быстрого спуска зависит от выбранной нормы и скалярного произведения. Дерево лишь приближает антиградиент, поэтому его добавление без подходящего шага не обязано уменьшать риск.

## 3. Потери и псевдоостатки

| Задача и потеря | $s=-\partial L/\partial z$ | Что означает |
|---|---|---|
| MSE, $\tfrac12(y-z)^2$ | $y-z$ | Величина обычной ошибки |
| MAE, $|y-z|$ | $\operatorname{sign}(y-z)$ | Направление исправления; в $z=y$ берут субградиент |
| Бинарная log-loss, $y\in\{0,1\}$, $L=-y\log p-(1-y)\log(1-p)$, $p=\sigma(z)$ | $y-p$ | Разность между меткой и вероятностью; $z$ — логит |

**Почему не всегда $y-z$?** Это антиградиент только для выбранной квадратичной потери (с указанным множителем $1/2$). При асимметричной потере из семинара завышение на 5 штрафуется в десять раз сильнее занижения на 5; антиградиенты равны $-50$ и $+5$. Обычные остатки $-5$ и $+5$ игнорировали бы этот приоритет.

MSE подходит, когда большие ошибки нужно штрафовать сильнее. При выбросах рассматривают Huber или MAE; для квантилей — квантильную потерю; для бинарной классификации — log-loss. Если функция недифференцируема, формула обычного градиента не действует в точках излома: требуется выбор субградиента или специальная процедура.

**Две разные квадратичные задачи:** исходный риск может быть log-loss, а дерево на очередном шаге обучают приближать числа $s_i$ по $\sum_i(s_i-h(x_i))^2$. Это способ подобрать направление в пространстве предсказаний, а не замена исходной цели. Затем шаг или прогнозы листьев можно подбирать по исходной потере.

In [ ]:
import numpy as np

y = np.array([10., 20., 30.])
pred = np.array([12., 15., 25.])
s = y - pred  # MSE: антиградиент без общего множителя 1/n
print("Текущий прогноз:", pred)
print("Псевдоостатки:", s)
print("Прогноз после точного шага:", pred + s)

## 4. Алгоритм и пример на Python

1. Выбрать $F_0$ (для MSE — среднее ответов).
2. Посчитать $s_i$ по **исходной** функции потерь при текущих прогнозах.
3. Обучить небольшое регрессионное дерево $h_m(x_i)\approx s_i$.
4. При необходимости найти $\gamma_m=\arg\min_\gamma\sum_i L(y_i,F_{m-1}(x_i)+\gamma h_m(x_i))$ или оптимизировать значения листьев.
5. Обновить $F_m=F_{m-1}+\eta\gamma_m h_m$, повторить и оценить качество на отложенной выборке.

Следующий код реализует только простой вариант для MSE с фиксированным $\eta$. Дерево обучается на остатках; это помогает увидеть сумму моделей без деталей промышленных библиотек.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
X = np.linspace(-3, 3, 160).reshape(-1, 1)
y = np.sin(2 * X[:, 0]) + 0.2 * X[:, 0] + rng.normal(0, 0.16, len(X))
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

initial = float(y_train.mean())
train_pred = np.full(len(X_train), initial)
val_pred = np.full(len(X_val), initial)
eta = 0.15
trees = []
history = []
for step in range(30):
    residual = y_train - train_pred
    tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=5, random_state=step)
    tree.fit(X_train, residual)
    trees.append(tree)
    train_pred += eta * tree.predict(X_train)
    val_pred += eta * tree.predict(X_val)
    history.append((mean_squared_error(y_train, train_pred),
                    mean_squared_error(y_val, val_pred)))

print(f"F0 = {initial:.3f}; после 30 деревьев: train MSE = {history[-1][0]:.3f}, validation MSE = {history[-1][1]:.3f}")

grid = np.linspace(-3.2, 3.2, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_train[:, 0], y_train, s=12, alpha=.5, label="Обучение")
for count in [0, 1, 5, 30]:
    curve = initial + eta * sum((tree.predict(grid) for tree in trees[:count]), np.zeros(len(grid)))
    axes[0].plot(grid[:, 0], curve, label=f"{count} деревьев")
axes[0].legend(); axes[0].set_title("Последовательные поправки")
axes[1].plot(np.arange(1, 31), history)
axes[1].legend(["train MSE", "validation MSE"])
axes[1].set_xlabel("Число деревьев"); axes[1].set_title("Ошибка ансамбля")
plt.tight_layout(); plt.show()

## 5. Глубина деревьев и контроль переобучения

Неглубокое дерево даёт простую поправку: оно захватывает несколько основных разбиений, а следующие деревья постепенно уточняют результат. Малое $\eta$ снижает влияние каждого шага; число деревьев и глубину подбирают по валидации, часто с ранней остановкой. Глубокое дерево может быстро подогнать остатки вместе с шумом и обычно дороже при обучении и предсказании.

Это **не правило «всегда глубина 2»**: для сложных взаимодействий нужны более глубокие деревья. Достаточная глубина зависит от данных, размера выборки, ограничения листьев и регуляризации. Для корректной оценки ранней остановки используют валидацию, а не тестовую выборку. Ансамбль деревьев также обычно не продолжает линейный тренд за пределы значений признака, встречавшихся при обучении.

## 6. Бустинг и случайный лес: сложность

Обозначим $n$ — число объектов, $d$ — число проверяемых признаков, $D$ — максимальная глубина дерева, $M$ — число деревьев. В идеализированной схеме после подготовки кандидатов поиск лучшего порога по одному признаку в узле занимает $O(n_{узла})$: статистики обновляют за один проход. На одном уровне сумма объектов по узлам не превышает $n$, поэтому стоимость дерева $O(Ddn)$.

| Операция | Бустинг | Случайный лес |
|---|---:|---:|
| Обучение, последовательная сумма работы | $O(MDdn)$ | $O(MDfn)$, где $f\le d$ признаков пробуют в узле |
| Прогноз одного объекта | $O(MD)$ | $O(MD)$ |
| Память на деревья | $O(M2^D)$ в грубой верхней оценке | $O(M2^D)$ |

Для леса bootstrap обычно берёт порядка $n$ строк на дерево; при $f=d$ и одинаковых $M,D$ **оценки общего объёма работы одного порядка**. Фактические $D$, $M$, $f$ и стоимость разбиения часто различаются. Лес может строить деревья параллельно, а бустинг зависит от предыдущих деревьев. Поэтому одинаковый Big O не означает одинаковое время работы.

**Условия оценки:** сортировка непрерывных значений или построение гистограмм добавляет собственную стоимость (например, первоначальная сортировка порядка $O(dn\log n)$ в простой схеме). Наивный пересчёт всех порогов и некоторые реализации дают другие оценки. Память $O(M2^D)$ — грубый верхний предел числа узлов, а не типичный размер модели.

## 7. XGBoost, LightGBM, CatBoost

Три популярные библиотеки реализуют деревья с градиентным бустингом, но используют разные инженерные решения. `scikit-learn` (`GradientBoostingRegressor`, `HistGradientBoostingRegressor`) удобен как учебная основа и базовая модель.

| Библиотека | Основной акцент | Практический ориентир |
|---|---|---|
| **XGBoost** | Регуляризация дерева и весов листьев, градиенты и вторые производные; гистограммный поиск разбиений (`hist`) | Гибкие настройки, в том числе регуляризация и работа с пропусками; начать с ограниченной глубины |
| **LightGBM** | Гистограммы и рост по листу с наибольшим улучшением (`leaf-wise`) | Удобен для больших табличных данных; на малой выборке следить за `num_leaves` и размером листа |
| **CatBoost** | Специальная обработка категориальных признаков и упорядоченные методы против смещения при использовании целевой статистики | Удобен при наличии категориальных столбцов; указывать их явно, следить за проверкой без утечки |

Ни одна из библиотек не является лучшей для всех наборов данных. Сравнивать следует на одинаковых разбиениях и метрике. 